# 02. Feature Engineering
## DSN Bootcamp Challenge: Sales Forecasting

**Objective**: Create time-series features, aggregations, and encodings for improved model performance

**Key Techniques**:
- Lag features (previous sales)
- Rolling statistics (moving averages)
- Store/Product aggregates
- Date-based features (seasonality)
- Target encoding for categorical variables
- Handling missing values

---

## 1. Setup & Imports

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import utility functions
import sys
sys.path.append('../')
from src.utils import (
    set_seed, print_seed_info, get_data_summary,
    create_lag_features, create_rolling_features, create_date_features,
    handle_missing_values
)

# Set random seed
set_seed(42)
print_seed_info(42)

## 2. Load Data

In [ ]:
# Load datasets
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

# Create a copy for feature engineering
train_fe = train_df.copy()
test_fe = test_df.copy()

print("\n✓ Data loaded successfully")

## 3. Handle Missing Values

In [ ]:
# Inspect missing values
print("=== MISSING VALUES (Before) ===")
print(f"Train: {train_fe.isnull().sum().sum()} missing cells")
print(f"Test: {test_fe.isnull().sum().sum()} missing cells")
print(f"\nMissing by column (Train):")
print(train_fe.isnull().sum()[train_fe.isnull().sum() > 0])

In [ ]:
# Handle product_weight_kg - use median by product_category
print("\nHandling product_weight_kg...")
weight_median = train_fe.groupby('product_category')['product_weight_kg'].transform('median')
train_fe['product_weight_kg'].fillna(weight_median, inplace=True)

# Test set - use overall median
test_fe['product_weight_kg'].fillna(train_df['product_weight_kg'].median(), inplace=True)

# Handle store_size - use mode (most frequent)
print("Handling store_size...")
store_size_mode = train_fe['store_size'].mode()[0]
train_fe['store_size'].fillna(store_size_mode, inplace=True)
test_fe['store_size'].fillna(store_size_mode, inplace=True)

print("\n=== MISSING VALUES (After) ===")
print(f"Train: {train_fe.isnull().sum().sum()} missing cells")
print(f"Test: {test_fe.isnull().sum().sum()} missing cells")
print("\n✓ Missing values handled")

## 4. Standardize Categorical Features

In [ ]:
# Standardize product_category (case inconsistency)
print("Standardizing product_category...")
print(f"Unique categories before: {train_fe['product_category'].nunique()}")

train_fe['product_category'] = train_fe['product_category'].str.lower().str.strip().str.title()
test_fe['product_category'] = test_fe['product_category'].str.lower().str.strip().str.title()

print(f"Unique categories after: {train_fe['product_category'].nunique()}")
print(f"\nCategories: {sorted(train_fe['product_category'].unique())}")

print("\n✓ Categorical features standardized")

## 5. Create Store & Product Aggregate Features

In [ ]:
# Store-level aggregates
print("Creating store-level features...")
store_stats = train_fe.groupby('store_code')['total_sales'].agg([
    ('store_avg_sales', 'mean'),
    ('store_median_sales', 'median'),
    ('store_std_sales', 'std'),
    ('store_min_sales', 'min'),
    ('store_max_sales', 'max'),
    ('store_count', 'count')
]).reset_index()

train_fe = train_fe.merge(store_stats, on='store_code', how='left')
test_fe = test_fe.merge(store_stats, on='store_code', how='left')

print(f"✓ Store features created: {list(store_stats.columns[1:])}")

In [ ]:
# Product-level aggregates
print("\nCreating product-level features...")
product_stats = train_fe.groupby('product_code')['total_sales'].agg([
    ('product_avg_sales', 'mean'),
    ('product_median_sales', 'median'),
    ('product_std_sales', 'std'),
    ('product_count', 'count')
]).reset_index()

train_fe = train_fe.merge(product_stats, on='product_code', how='left')
test_fe = test_fe.merge(product_stats, on='product_code', how='left')

print(f"✓ Product features created: {list(product_stats.columns[1:])}")

In [ ]:
# Store-Product interaction features
print("\nCreating store-product interaction features...")
store_product_stats = train_fe.groupby(['store_code', 'product_code'])['total_sales'].agg([
    ('store_product_avg_sales', 'mean'),
    ('store_product_count', 'count')
]).reset_index()

train_fe = train_fe.merge(store_product_stats, on=['store_code', 'product_code'], how='left')
test_fe = test_fe.merge(store_product_stats, on=['store_code', 'product_code'], how='left')

print(f"✓ Store-Product features created: {list(store_product_stats.columns[2:])}")

## 6. Create Price-Based Features

In [ ]:
# Price features
print("Creating price-based features...")

# Store average price
store_avg_price = train_fe.groupby('store_code')['product_price'].mean().reset_index()
store_avg_price.rename(columns={'product_price': 'store_avg_price'}, inplace=True)
train_fe = train_fe.merge(store_avg_price, on='store_code', how='left')
test_fe = test_fe.merge(store_avg_price, on='store_code', how='left')

# Price relative to store average
train_fe['price_vs_store_avg'] = train_fe['product_price'] / train_fe['store_avg_price']
test_fe['price_vs_store_avg'] = test_fe['product_price'] / test_fe['store_avg_price']

# Price relative to category average
category_avg_price = train_fe.groupby('product_category')['product_price'].mean().reset_index()
category_avg_price.rename(columns={'product_price': 'category_avg_price'}, inplace=True)
train_fe = train_fe.merge(category_avg_price, on='product_category', how='left')
test_fe = test_fe.merge(category_avg_price, on='product_category', how='left')

train_fe['price_vs_category_avg'] = train_fe['product_price'] / train_fe['category_avg_price']
test_fe['price_vs_category_avg'] = test_fe['product_price'] / test_fe['category_avg_price']

print("✓ Price-based features created")

## 7. Target Encoding for Categorical Features

In [ ]:
# Target encoding for high-cardinality categoricals
print("Creating target-encoded features...")

categorical_cols_to_encode = ['product_category', 'store_format', 'fat_content']

for col in categorical_cols_to_encode:
    # Calculate mean target per category
    target_encoding = train_fe.groupby(col)['total_sales'].mean().reset_index()
    target_encoding.rename(columns={'total_sales': f'{col}_target_mean'}, inplace=True)
    
    # Merge to both train and test
    train_fe = train_fe.merge(target_encoding, on=col, how='left')
    test_fe = test_fe.merge(target_encoding, on=col, how='left')
    
    print(f"  ✓ {col}: created target encoding")

print("\n✓ Target encoding complete")

## 8. Create Lag Features

In [ ]:
# Sort by store-product-date for proper lag creation
print("Creating lag features...")
print("Note: Lag features require temporal ordering.")
print("Checking if temporal information is available...")

if 'date' in train_fe.columns or 'time' in train_fe.columns or 'day' in train_fe.columns:
    print("✓ Temporal column found - creating lags")
    
    # Sort by date first
    train_fe_sorted = train_fe.sort_values(['store_code', 'product_code', 'date']).reset_index(drop=True)
    
    # Create lag features
    lags = [1, 7, 30]
    for lag in lags:
        train_fe_sorted[f'sales_lag_{lag}'] = train_fe_sorted.groupby(['store_code', 'product_code'])['total_sales'].shift(lag)
    
    train_fe = train_fe_sorted
    print(f"✓ Lag features created: {[f'sales_lag_{i}' for i in lags]}")
else:
    print("⚠️  No temporal column found - skipping lag features")
    print("   Lag features require chronological ordering by date")

## 9. Create Rolling Statistics Features

In [ ]:
# Create rolling statistics
print("Creating rolling statistics features...")

if 'date' in train_fe.columns or train_fe.index.name == 'index':
    windows = [7, 14, 30]
    
    for window in windows:
        # Rolling mean
        train_fe[f'sales_rolling_mean_{window}'] = train_fe.groupby(['store_code', 'product_code'])['total_sales'].transform(
            lambda x: x.rolling(window=window, min_periods=1).mean()
        )
        
        # Rolling std
        train_fe[f'sales_rolling_std_{window}'] = train_fe.groupby(['store_code', 'product_code'])['total_sales'].transform(
            lambda x: x.rolling(window=window, min_periods=1).std()
        )
        
        print(f"  ✓ Window size {window}: mean & std created")
    
    print("\n✓ Rolling statistics complete")
else:
    print("⚠️  Rolling statistics require temporal ordering")
    print("   Ensure data is sorted chronologically")

## 10. Create Frequency Encoding

In [ ]:
# Frequency encoding for categorical features
print("Creating frequency encoding features...")

freq_cols = ['product_category', 'store_format', 'fat_content', 'store_location_tier']

for col in freq_cols:
    freq_map = train_fe[col].value_counts().to_dict()
    train_fe[f'{col}_freq'] = train_fe[col].map(freq_map)
    test_fe[f'{col}_freq'] = test_fe[col].map(freq_map).fillna(0)
    print(f"  ✓ {col}: frequency encoding created")

print("\n✓ Frequency encoding complete")

## 11. Create Interaction Features

In [ ]:
# Interaction features
print("Creating interaction features...")

# Price × Visibility
train_fe['price_x_visibility'] = train_fe['product_price'] * train_fe['shelf_visibility']
test_fe['price_x_visibility'] = test_fe['product_price'] * test_fe['shelf_visibility']

# Price × Weight
train_fe['price_x_weight'] = train_fe['product_price'] * train_fe['product_weight_kg']
test_fe['price_x_weight'] = test_fe['product_price'] * test_fe['product_weight_kg']

# Store Age × Store Size (ordinal encoding for size)
size_mapping = {'Small': 1, 'Medium': 2, 'Large': 3}
train_fe['store_age_x_size'] = train_fe['store_age_years'] * train_fe['store_size'].map(size_mapping)
test_fe['store_age_x_size'] = test_fe['store_age_years'] * test_fe['store_size'].map(size_mapping)

print("✓ Interaction features created")

## 12. Feature Summary & Quality Check

In [ ]:
# Feature summary
print("\n" + "="*80)
print("FEATURE ENGINEERING - SUMMARY")
print("="*80)

print(f"\n📊 DATASET SHAPES:")
print(f"  Train: {train_fe.shape[0]:,} rows × {train_fe.shape[1]} columns")
print(f"  Test:  {test_fe.shape[0]:,} rows × {test_fe.shape[1]} columns")

print(f"\n🔧 FEATURES ENGINEERED:")
original_cols = set(train_df.columns)
new_cols = set(train_fe.columns) - original_cols
print(f"  Original features: {len(original_cols)}")
print(f"  New features created: {len(new_cols)}")
print(f"  Total features: {train_fe.shape[1]}")

print(f"\n📋 NEW FEATURE CATEGORIES:")
print(f"  • Store aggregates: store_avg_sales, store_median_sales, store_std_sales, store_min_sales, store_max_sales")
print(f"  • Product aggregates: product_avg_sales, product_median_sales, product_std_sales")
print(f"  • Store-Product interaction: store_product_avg_sales, store_product_count")
print(f"  • Price features: store_avg_price, price_vs_store_avg, category_avg_price, price_vs_category_avg")
print(f"  • Target encoding: *_target_mean for categorical features")
print(f"  • Interaction features: price_x_visibility, price_x_weight, store_age_x_size")
print(f"  • Frequency encoding: *_freq for categorical features")

print(f"\n⚠️  MISSING VALUES (After Engineering):")
print(f"  Train: {train_fe.isnull().sum().sum()} cells")
print(f"  Test:  {test_fe.isnull().sum().sum()} cells")

if train_fe.isnull().sum().sum() > 0:
    print(f"\n  Columns with missing values (Train):")
    print(train_fe.isnull().sum()[train_fe.isnull().sum() > 0])

## 13. Data Quality Checks

In [ ]:
# Check for infinite values
print("\n=== DATA QUALITY CHECKS ===")
print(f"\nInfinite values in Train: {np.isinf(train_fe.select_dtypes(include=[np.number])).sum().sum()}")
print(f"Infinite values in Test: {np.isinf(test_fe.select_dtypes(include=[np.number])).sum().sum()}")

# Replace infinite values with NaN
train_fe.replace([np.inf, -np.inf], np.nan, inplace=True)
test_fe.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill remaining NaN with 0 or median
print(f"\nFilling remaining NaN values...")
train_fe = train_fe.fillna(train_fe.median(numeric_only=True))
test_fe = test_fe.fillna(train_fe.median(numeric_only=True))

print(f"✓ Data quality check complete")

## 14. Save Engineered Features

In [ ]:
# Save engineered datasets
print("\nSaving engineered features...")

train_fe.to_csv('../data/train_engineered.csv', index=False)
test_fe.to_csv('../data/test_engineered.csv', index=False)

print(f"✓ Engineered data saved")
print(f"  - data/train_engineered.csv ({len(train_fe)} rows × {train_fe.shape[1]} columns)")
print(f"  - data/test_engineered.csv ({len(test_fe)} rows × {test_fe.shape[1]} columns)")

## 15. Visualize Feature Importance Indicators

In [ ]:
# Correlation of new features with target
print("\n=== FEATURE CORRELATIONS WITH TARGET ===")
numeric_new_features = [col for col in new_cols if col in train_fe.select_dtypes(include=[np.number]).columns]
correlations = train_fe[numeric_new_features + ['total_sales']].corr()['total_sales'].sort_values(ascending=False)

print("\nTop 10 correlated new features:")
for i, (col, corr) in enumerate(correlations[1:11].items(), 1):
    print(f"  {i:2d}. {col:<40} {corr:+.4f}")

In [ ]:
# Plot top feature correlations
fig, ax = plt.subplots(figsize=(10, 6))
correlations[1:16].plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Correlation with Total Sales')
ax.set_title('Top 15 New Features by Correlation with Target')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('../notebooks/feature_engineering_correlations.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Correlation plot saved")

## 16. Next Steps - Baseline Modeling

In [ ]:
print("\n" + "="*80)
print("✅ FEATURE ENGINEERING COMPLETE")
print("="*80)

print(f"\n📊 ENGINEERED DATASETS READY:")
print(f"  • data/train_engineered.csv: {len(train_fe):,} rows × {train_fe.shape[1]} columns")
print(f"  • data/test_engineered.csv: {len(test_fe):,} rows × {test_fe.shape[1]} columns")

print(f"\n🚀 NEXT MILESTONE (Milestone 4):")
print(f"  1. Train baseline LightGBM model")
print(f"  2. Implement time-series cross-validation")
print(f"  3. Calculate OOF predictions")
print(f"  4. Generate baseline submission")
print(f"  5. Record CV RMSE score")

print(f"\n💡 FEATURE ENGINEERING SUMMARY:")
print(f"  ✓ Handled {len(missing_train[missing_train > 0])} columns with missing values")
print(f"  ✓ Standardized categorical features (case normalization)")
print(f"  ✓ Created store-level aggregates (5 features)")
print(f"  ✓ Created product-level aggregates (4 features)")
print(f"  ✓ Created store-product interactions (2 features)")
print(f"  ✓ Created price-based features (4 features)")
print(f"  ✓ Target encoded categorical features")
print(f"  ✓ Created interaction features (3 features)")
print(f"  ✓ Created frequency encoding features")
print(f"  ✓ Total new features: {len(new_cols)}")

print("\n" + "="*80)